# Setup

In [ ]:
# TERMINAL -> alternatywnie https://github.com/InfuseAI/colab-xterm

# curl -fsSL https://ollama.com/install.sh | PATH="/sbin:/usr/sbin:$PATH" sh

# ollama serve &

# ollama pull llama3.1:latest

In [1]:
!pip -q install ollama

In [2]:
import ollama
import requests

In [ ]:
class CFG:
    model = "llama3.1:latest"

# Funkcje

In [ ]:
def get_current_weather(city):
    base_url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(base_url)
    data = response.json()
    return f"Temp in {city}: {data['current_condition'][0]['temp_C']}"

In [ ]:
def chat_with_ollama_no_functions(user_question):
    response = ollama.chat(
        model=CFG.model, messages=[{"role": "user", "content": user_question}]
    )
    return response

In [ ]:
def chat_with_ollama(user_question):
    response = ollama.chat(
        model=CFG.model,
        messages=[{"role": "user", "content": user_question}],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "get_current_weather",
                    "description": "Get the current weather for a city",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "city": {
                                "type": "string",
                                "description": "City",
                            },
                        },
                        "required": ["city"],
                    },
                },
            },
        ],
    )
    return response

Ta funkcja `chat_with_ollama` jest rozszerzeniem poprzedniej funkcji i wprowadza bardzo ważną koncepcję - narzędzia zewnętrzne (tools) dla modelu językowego. Przeanalizujmy szczegółowo jej działanie:

1. Podobnie jak poprzednia funkcja, przyjmuje ona parametr `user_question` - pytanie użytkownika do modelu.

2. Ponownie używamy metody `ollama.chat()`, ale tym razem przekazujemy dodatkowy parametr `tools`, który definiuje funkcje, jakie model może wywołać podczas generowania odpowiedzi.

3. W ramach parametru `tools` definiujemy listę dostępnych narzędzi. W tym przypadku mamy tylko jedno narzędzie typu `'function'`, czyli funkcję, którą model może wywołać.

4. Ta funkcja jest opisana przez kilka elementów:
   - `'name'`: nazwa funkcji, którą model będzie mógł wywołać - "get_current_weather". Jest to dokładnie ta sama funkcja, którą zdefiniowaliśmy wcześniej.
   - `'description'`: opis funkcji, pomagający modelowi zrozumieć, kiedy powinien z niej skorzystać - "Get the current weather for a city".
   - `'parameters'`: definicja parametrów, jakie funkcja przyjmuje, w formacie zgodnym ze schematem JSON.

5. W ramach `'parameters'` określamy, że:
   - Funkcja przyjmuje obiekt (`'type': "object"`).
   - Obiekt ten ma właściwość `'city'` typu string, służącą do określenia miasta.
   - Parametr `'city'` jest wymagany (`'required': ['city']`).

6. Na końcu zwracamy otrzymaną odpowiedź od modelu, podobnie jak w poprzedniej funkcji.

Ta funkcja wprowadza kluczowy mechanizm - możliwość "rozszerzenia" wiedzy modelu językowego o aktualne dane z zewnętrznych źródeł. Gdy model stwierdzi, że do odpowiedzi na pytanie użytkownika potrzebuje aktualnych danych pogodowych, nie będzie próbował "zgadywać" tych informacji, tylko wywoła funkcję `get_current_weather`, podając jako argument odpowiednie miasto.

# Skrypt


In [ ]:
while True:
    user_input = input("Enter your question (or 'quit' to exit): ")
    if user_input.lower() == "quit":
        break

    response = chat_with_ollama(user_input)

    if "tool_calls" in response["message"] and response["message"]["tool_calls"]:
        tools_calls = response["message"]["tool_calls"]
        for tool_call in tools_calls:
            tool_name = tool_call["function"]["name"]
            arguments = tool_call["function"]["arguments"]

            if tool_name == "get_current_weather" and "city" in arguments:
                result = get_current_weather(arguments["city"])
                print("Weather function result:", result)

    else:
        # If no tool calls or no valid arguments, use the LLM's response
        response = chat_with_ollama_no_functions(user_input)
        print("AI response:", response["message"]["content"])


Enter your question (or 'quit' to exit): what is the temperature in Haarlem?
Weather function result: Temp in Haarlem: 12
Enter your question (or 'quit' to exit): And what about Warsaw?
Weather function result: Temp in Warsaw: 11
Enter your question (or 'quit' to exit): quit
